# Forward and Back Substitution

Chapter 5: Square Matrix and LU Decomposition · *Mathematics for AI and Machine Learning* by Fuheng Wu

Solve Ly=b then Ux=y and compare with direct solve.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-5/forward-backward/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

# The demo's default sliders: unit lower L, upper U and right-hand side b
L = torch.tensor([[1.0, 0.0],
                  [1.1, 1.0]], dtype=torch.float64)
U = torch.tensor([[2.2, 0.6],
                  [0.0, 1.7]], dtype=torch.float64)
b = torch.tensor([1.0, 2.5], dtype=torch.float64)


def forward_sub(L, b):
    """Solve L y = b top to bottom."""
    y = torch.zeros_like(b)
    for i in range(len(b)):
        y[i] = (b[i] - L[i, :i] @ y[:i]) / L[i, i]
    return y


def back_sub(U, y):
    """Solve U x = y bottom to top."""
    x = torch.zeros_like(y)
    for i in reversed(range(len(y))):
        x[i] = (y[i] - U[i, i + 1:] @ x[i + 1:]) / U[i, i]
    return x


y = forward_sub(L, b)
x = back_sub(U, y)
A = L @ U
print("y (L y = b) =", y.tolist())
print("x (U x = y) =", x.tolist())
print("direct solve A x = b:", torch.linalg.solve(A, b).tolist())
print("residual ||A x - b|| =", (A @ x - b).norm().item())

# ── The library triangular solver does the same two sweeps ──────────────────
y2 = torch.linalg.solve_triangular(L, b.unsqueeze(1), upper=False, unitriangular=True)
x2 = torch.linalg.solve_triangular(U, y2, upper=True)
print("solve_triangular x =", x2.squeeze(1).tolist())
# Factor once (O(n^3)), then every new b costs only two O(n^2) sweeps.

y (L y = b) = [1.0, 1.4]
x (U x = y) = [0.2299465240641711, 0.8235294117647058]
direct solve A x = b: [0.22994652406417107, 0.8235294117647061]
residual ||A x - b|| = 0.0
solve_triangular x = [0.22994652406417115, 0.8235294117647058]
